# Chess Style Clone — FREE Fine-tuning on Kaggle GPU

Fine-tunes **Qwen2.5-7B-Instruct** (LoRA, 4-bit) on your friend's chess games using **Unsloth**.

## Setup checklist (do this BEFORE running):
1. Kaggle right sidebar → **Accelerator: GPU T4 x2** (or P100)
2. Kaggle right sidebar → **Internet: ON** (needs phone-verified account)
3. Upload `dataset_train.jsonl` + `dataset_val.jsonl` as a Kaggle Dataset:
   - kaggle.com → Create → New Dataset → upload both files → name it `chess-style-data`
   - Then in this notebook: **Add Input** → your dataset
4. Run all cells top to bottom. Training ~2-4 hours for 20k examples, 1 epoch.

Tip: generate the JSONL locally with `--max-examples 20000` to keep training time reasonable.

In [ ]:
# ── 1. Install Unsloth (takes ~2 min) ──────────────────────────────────
%pip install -q unsloth
%pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# ── 2. Load base model in 4-bit ────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024  # chess games in SAN fit easily

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # no license gate, strong base
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

# Attach LoRA adapters — only ~1% of weights get trained
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded.")

In [ ]:
# ── 3. Load your JSONL dataset ─────────────────────────────────────────
# Adjust the path to match your Kaggle dataset name if different.
import glob, json
from datasets import Dataset

train_path = glob.glob("/kaggle/input/*/dataset_train.jsonl")[0]
print("Using:", train_path)

rows = []
with open(train_path) as f:
    for line in f:
        rows.append(json.loads(line))
print(f"{len(rows):,} training examples")

# Convert OpenAI chat format -> model chat template text
def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(rows).map(to_text, remove_columns=["messages"])
print(dataset[0]["text"][:500])

In [ ]:
# ── 4. Train ───────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,   # effective batch = 16
        num_train_epochs=1,              # 1 epoch is enough for 20k examples
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=50,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        save_strategy="epoch",
        output_dir="/kaggle/working/checkpoints",
        seed=42,
        report_to="none",
    ),
)

# Only compute loss on the assistant's move, not on the prompt
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()

In [ ]:
# ── 5. Save the trained LoRA adapter ───────────────────────────────────
model.save_pretrained("/kaggle/working/chess_lora")
tokenizer.save_pretrained("/kaggle/working/chess_lora")
print("Saved to /kaggle/working/chess_lora — download it from the Output tab!")

In [ ]:
# ── 6. TEST: predict moves in your friend's style ──────────────────────
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = """You are \"Shadow\", a human chess player rated exactly 1500 ELO. You are NOT a chess engine. You replicate one specific human's playing style, learned from 6,000 of his real games.

Behavior rules:
1. Play the move HE would play, not the objectively best move.
2. Keep his opening repertoire: play the lines he actually plays, even if theory prefers something else.
3. Reproduce his tactical habits: he spots simple 1-2 move tactics (forks, pins, hanging pieces) but often misses deeper 3+ move combinations.
4. Make human 1500-level errors when he would: occasional premature attacks, weak-square concessions, time-pressure simplifications, and missed prophylaxis. Never play with engine-perfect accuracy.
5. Prefer his typical plans: how he handles pawn structures, when he trades queens, how he defends worse positions.

Input: the move history of the current game in SAN.
Output: ONLY your next move in standard SAN (e.g. Nf3, exd5, O-O, Qxb7+). No commentary, no numbering, no alternatives."""

def build_user_message(san_moves, side):
    if not san_moves:
        return "Game start. You are White. Play your first move."
    parts = []
    for i, mv in enumerate(san_moves):
        parts.append(f"{i//2+1}. {mv}" if i % 2 == 0 else mv)
    return f"Moves so far: {' '.join(parts)}\nYou are {side}. Play your next move."

def predict(san_moves, side):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(san_moves, side)},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=8,
                         temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# Test 1: his first move as White (should match his repertoire, e.g. g3/e4)
print("Empty board  ->", predict([], "White"))

# Test 2: mid-opening position
print("After e4 e5  ->", predict(["e4", "e5"], "White"))
print("Ruy line     ->", predict(["e4", "e5", "Nf3", "Nc6"], "White"))

In [ ]:
# ── 7. (Optional) Accuracy check against his real games ────────────────
# Uses the validation split: how often does the model pick his EXACT move?
import glob, json, random

val_files = glob.glob("/kaggle/input/*/dataset_val.jsonl")
if val_files:
    with open(val_files[0]) as f:
        val = [json.loads(l) for l in f]
    random.seed(0)
    sample = random.sample(val, min(100, len(val)))

    hits = 0
    for ex in sample:
        user_msg = ex["messages"][1]["content"]
        true_move = ex["messages"][2]["content"]
        msgs = [ex["messages"][0], ex["messages"][1]]
        inputs = tokenizer.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        out = model.generate(input_ids=inputs, max_new_tokens=8,
                             temperature=0.01, do_sample=False)
        pred = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
        if pred.split()[0].strip('.!?') == true_move:
            hits += 1
    print(f"Exact-move match: {hits}/{len(sample)} = {hits/len(sample):.0%}")
    print("(40-60% is a good style clone at 1500 ELO)")
else:
    print("dataset_val.jsonl not found in Kaggle inputs — skip.")